# Прогноз времени на ДЗ — доработка

Сейчас ноутбук **не запускается**. Я отметил 3 ошибки-блокера и пару замечаний. Найди и исправь сам, потом запусти.

In [ ]:
import gradio as gr
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split


np.random.seed(42)
n_samples = 100

subjects = np.random.randint(1, 7, size=n_samples)
difficulty = np.random.randint(1, 6, size=n_samples)


# 🔴 ОШИБКА 1 (синтаксис): "+1 np.random..." — между 1 и np нет оператора.
# Что ты хотел прибавить? Скорее всего просто шум np.random.normal(...).
# Лишняя 1 тут откуда? Убери её и поставь правильный "+".
time_spent = subjects * 25 + difficulty * 15 +1 np.random.normal(0, 10, n_samples)
time_spent = np.clip(time_spent, 15, 300)

X = pd.DataFrame({"subjects": subjects, "difficulty": difficulty})
y = time_spent


X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)


model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
baseline_pred = np.full_like(y_test, y_train.mean())

mae_baseline = mean_absolute_error(y_test, baseline_pred)
mae_model = mean_absolute_error(y_test, y_pred)

print(f"Baseline MAE: {mae_baseline:.2f} min")
print(f"Model MAE: {mae_model:.2f} min")


# 🔴 ОШИБКА 2 (отступы): тело функции должно быть С ОТСТУПОМ под def.
# Сейчас input_data, prediction и return на одном уровне с def -> IndentationError.
# Добавь 4 пробела перед строками тела функции.
def predict(subjects_count, difficulty_level):
input_data = pd.DataFrame(
[[subjects_count, difficulty_level]],
columns=["subjects", "difficulty"],
)
prediction = model.predict(input_data)[0]
return f"{max(10, int(round(prediction)))} минут"


demo = gr.Interface(
fn=predict,
inputs=[
gr.Slider(minimum=1, maximum=6, step=1, value=3, label="Количество предметов"),
gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Уровень сложности (1-5)"),
],
outputs=gr.Textbox(label="Прогнозируемое время на ДЗ"),
title="Прогноз времени на домашнее задание",
description="Модель машинного обучения на базе Linear Regression",
)

# 🔴 ОШИБКА 3: "if name == \"main\":" неправильно.
# 1) переменная называется __name__, а значение "__main__" (по два подчёркивания).
# 2) demo.launch() под if должен быть с отступом.
# 3) ПОДСКАЗКА: в Colab __name__ НЕ равен "__main__", поэтому этот if лишний —
#    проще убрать его и просто вызвать demo.launch().
if name == "main":
demo.launch()